# Chapter 2: Introduction to the Relational Model


## Core Question

A table is more than a grid. To query and connect tables correctly, we must know what a
row represents, which attributes identify it, how tables refer to one another, and how a
small set of relational operations describes a query.


## Teaching Summary

| Topic | Worked example and practice | Evidence to retain |
|---|---|---|
| Relation, tuple, attribute, domain, schema, and instance | Identify parts of course-registration relations | Schema identification sheet |
| Primary, candidate, and foreign keys | Compare candidate keys and follow references | Key map with reasons |
| Core relational algebra | Apply selection, projection, product, join, and set operations | Result after each operation |

Chapter 3 expresses these query ideas in SQL. This chapter focuses on structure and query
logic rather than SQL syntax. Assignment, rename, and formal equivalence proofs are
extensions and are not major Exam 1 operations.


## Connection to Chapter 3

The relational model supplies the vocabulary and operations that SQL implements. The
keys and foreign-key paths in this chapter become join conditions, while selection and
projection become common parts of a `SELECT` query.


## Prerequisites

- Read a two-dimensional table.
- Understand that a mathematical set contains one copy of each element.
- Apply equality, inequality, `AND`, and `OR` conditions.
- Follow an explicit sequence of filtering and column-selection steps.


## Learning Objectives

After completing this chapter, you should be able to:

1. Identify a relation, tuple, attribute, domain, schema, and relation instance.
2. Explain the difference between a schema and an instance.
3. Classify superkeys, candidate keys, primary keys, and composite keys from business
   rules.
4. Identify referencing and referenced relations for a foreign key.
5. Read a schema diagram and follow foreign-key connections.
6. Apply selection, projection, Cartesian product, theta join, union, intersection, and
   set difference to small relation instances.
7. Compare the purpose of two simple relational-algebra expressions.


## Course-Registration Data

### `department`

| dept_code | dept_name | building |
|---|---|---|
| DES | Digital Design | Hong Hall |
| FIN | Finance | Cheng Hall |
| IM | Information Management | Hong Hall |

### `student`

| student_id | email | student_name | dept_code |
|---|---|---|---|
| S101 | an.chen@example.edu | An Chen | IM |
| S102 | bea.lin@example.edu | Bea Lin | FIN |
| S103 | kai.wu@example.edu | Kai Wu | IM |
| S104 | mira.ho@example.edu | Mira Ho | DES |

### `course`

| course_id | title | dept_code | credits |
|---|---|---|---:|
| DB201 | Database Management | IM | 3 |
| FT210 | Financial Technology | FIN | 3 |
| ML230 | Machine Learning | IM | 3 |
| WD120 | Web Design | DES | 2 |

### `enrollment`

| student_id | course_id | term | grade |
|---|---|---|---|
| S101 | DB201 | 115-1 | A |
| S101 | FT210 | 115-1 | B+ |
| S102 | FT210 | 115-1 | A- |
| S103 | DB201 | 115-1 | B |
| S103 | ML230 | 115-1 | A |
| S104 | WD120 | 115-1 | A- |

The design uses these business rules:

- Each student has one unique, stable `student_id`.
- Each email belongs to at most one student; names may repeat.
- Each department and course has a unique code.
- A student has at most one enrollment in the same course and term.
- Student and course department codes must reference an existing department.
- Enrollment student and course identifiers must reference existing rows.


## 1. Relations, Tuples, Attributes, and Domains

- A **relation** corresponds to a table.
- A **tuple** corresponds to one row.
- An **attribute** corresponds to one column.
- A **relation instance** is the set of tuples stored at a particular time.
- A **domain** is the set of values allowed for an attribute.

For `student`, `student_id` is an attribute and
`(S101, an.chen@example.edu, An Chen, IM)` is a tuple. The four current rows form the
current relation instance.

### Worked Example

`course` has four attributes and four current tuples. Adding a course changes the
instance. Adding an `admission_year` attribute changes the schema.

### Predict and Check

In `enrollment`, identify the relation name, all attributes, and the tuple representing
S103 taking ML230. Separate attribute names from attribute values.

### Atomic Values, Order, and Duplicates

A value is atomic when the design treats it as one indivisible value. Storing several
phone numbers in one cell makes independent phone operations difficult. A clearer design
is `student_phone(student_id, phone_number)` with one phone number per row.

In the formal relational model, a relation is a set. Tuple display order is not part of
the relation, and identical duplicate tuples are not retained. SQL tables may permit
duplicates; Chapter 3 revisits this difference with `DISTINCT`.

Predict whether rearranging the four `student` rows changes the formal relation. It does
not. Required display order must be expressed by a query.


## 2. Schema and Instance

A **relation schema** describes a relation's name, attributes, domains, and constraints.
An **instance** is the current data.

```text
student(student_id, email, student_name, dept_code)
```

Changing S102 from department FIN to IM changes the instance. Adding a new attribute
changes the schema and requires a decision about values for existing rows.


## 3. Keys

Keys follow from the schema and business rules, not merely from accidental uniqueness in
the current sample.

- A **superkey** is an attribute set that uniquely identifies a tuple. It may contain
  unnecessary attributes.
- A **candidate key** is a minimal superkey.
- A **primary key** is the candidate key selected as the main identifier.
- A **composite key** contains more than one attribute.

### Worked Example: `student`

`{student_id}` and `{email}` are candidate keys under the stated rules.
`{student_id, student_name}` is a superkey but not a candidate key because
`student_name` is unnecessary. `{student_name}` is not guaranteed to be unique even
though the four sample names differ. This design selects `student_id` as the primary key.

### Worked Example: `enrollment`

Neither `student_id` nor `course_id` alone identifies an enrollment. The composite key
`{student_id, course_id, term}` does under the stated rule.

### Practice

Compare `student_name`, `email`, and `student_id` as proposed primary keys. First identify
the candidate keys. Then discuss stability, possible changes, length, and business
meaning. A design conclusion requires reasons, not a vote.


## 4. Foreign Keys and Schema Diagrams

A **foreign key** appears in the referencing relation and must match a key in the
referenced relation. For example, `student.dept_code` references
`department.dept_code`.

Adding a student with department `LAW` violates the rule unless the LAW department is
created first. A foreign key need not be unique in the referencing relation; many
students may belong to IM.

```text
department
  PK dept_code
     dept_name
     building
       ^
       | student.dept_code, course.dept_code

student                              course
  PK student_id                        PK course_id
  CK email                             FK dept_code -> department.dept_code
     student_name                         title
  FK dept_code -> department.dept_code    credits
       ^                                  ^
       |                                  |
       +---------- enrollment ------------+
                    PK/FK student_id -> student.student_id
                    PK/FK course_id  -> course.course_id
                    PK    term
                          grade
```

To find the name and course department for enrollment `(S101, DB201, 115-1, A)`, follow
`enrollment.student_id` to Student, then `enrollment.course_id` to Course, and finally
`course.dept_code` to Department.

### Evidence to Retain

Create a schema-and-key sheet containing all primary keys, every foreign key and its
direction, the two Student candidate keys, one nonminimal superkey, and one modification
that would violate referential integrity.


### Relational schema diagram



This original diagram applies the chapter concepts to the synthetic course-registration example used throughout the notebooks.


## 5. Relational-Algebra Operations

Each relational-algebra operation accepts one or two relations and returns a relation.
This closure property allows operations to be composed.

| Operation | Symbol | Main question |
|---|---|---|
| Selection | `σ` | Which tuples remain? |
| Projection | `Π` | Which attributes remain? |
| Cartesian product | `×` | What are all cross-relation tuple combinations? |
| Theta join | `⋈_θ` | Which combinations satisfy the join condition? |
| Union | `∪` | Which tuples occur in either input? |
| Intersection | `∩` | Which tuples occur in both inputs? |
| Set difference | `−` | Which tuples occur in the left input but not the right? |

### Selection

```text
σ_dept_code='IM'(student)
```

This keeps the complete S101 and S103 tuples. Predict the result of
`σ_dept_code='IM' AND student_id!='S101'(student)` and explain which predicate excludes
each removed tuple.

### Projection

```text
Π_dept_code(student)
```

The formal result is `{DES, FIN, IM}`. Projection removes duplicate tuples because a
formal relation is a set. Predict `Π_building(department)` and handle the repeated
`Hong Hall` value correctly.

### Composition

```text
Π_student_name(σ_dept_code='IM'(student))
```

The inner selection keeps S101 and S103; the outer projection produces
`{An Chen, Kai Wu}`. Write an expression for the identifiers and titles of three-credit
courses, and state which operation runs first in the expression.

### Cartesian Product

For `{S101, S102} × {DB201, FT210}`, the result has four pairs. These are possible
combinations, not four enrollment facts. The complete Student and Course relations have
four tuples each, so their product has 16 tuples.

### Theta Join

```text
student ⋈_student.student_id=enrollment.student_id enrollment
```

A theta join can be understood as a Cartesian product followed by a selection:

```text
r ⋈_theta s = σ_theta(r × s)
```

Projecting `student_name` and `course_id` after the join produces the six actual
enrollment pairs. Omitting the join predicate produces 24 combinations, most of which
are not enrollment facts.

Practice: identify the two relations and join predicate needed to connect a course title
to its department name.


### Selection and projection pipeline



This original diagram applies the chapter concepts to the synthetic course-registration example used throughout the notebooks.


## 6. Set Operations

Union, intersection, and difference require compatible input relations: the same number
of attributes and compatible domains in corresponding positions.

Let:

```text
A = students in DB201 = {S101, S103}
B = students in FT210 = {S101, S102}
```

| Expression | Meaning | Result |
|---|---|---|
| `A ∪ B` | In DB201 or FT210 or both | `{S101, S102, S103}` |
| `A ∩ B` | In both courses | `{S101}` |
| `A − B` | In DB201 but not FT210 | `{S103}` |
| `B − A` | In FT210 but not DB201 | `{S102}` |

Set difference has direction. `student ∪ course` is invalid because the schemas are not
compatible.

Practice: let C be the students in ML230. Write C, then calculate `A ∪ C`, `A ∩ C`, and
`C − A`.


## Extensions

### Assignment

Assignment names an intermediate relation without changing the permanent database:

```text
db_students <- Π_student_id(σ_course_id='DB201'(enrollment))
fintech_students <- Π_student_id(σ_course_id='FT210'(enrollment))
db_students ∩ fintech_students
```

### Rename

Rename distinguishes multiple uses of the same relation:

```text
ρ_s1(student)
ρ_s2(student)
```

It supports a self-comparison such as finding different students in the same department.

### Simple Equivalence

These expressions place the same Student-only filter before or after an inner join:

```text
Q1 = σ_student.dept_code='IM'(
       student ⋈_student.student_id=enrollment.student_id enrollment
     )

Q2 = (σ_dept_code='IM'(student))
     ⋈_student.student_id=enrollment.student_id enrollment
```

They express the same result under the stated conditions. Equality on one sample alone is
not a proof for every legal instance. Chapter 16 revisits equivalence as an optimization
guardrail.


## Common Errors

1. Choosing a key from accidental uniqueness in the sample.
2. Calling every superkey a candidate key.
3. Drawing a foreign-key arrow in the wrong direction.
4. Assuming tuple display order is part of a relation.
5. Forgetting duplicate removal in formal projection and set operations.
6. Treating a Cartesian product as a factual relationship.
7. Omitting the join predicate.
8. Reversing set difference.


## Discussion and Individual Evidence

For each proposed key or algebra result, state the business rule or operation that
supports the answer. Retain the schema sheet, key map, each intermediate relation, and
one corrected misconception.


## Chapter Summary

A schema defines structure and constraints; an instance contains current tuples. Keys
identify rows and foreign keys connect relations. Selection filters tuples, projection
keeps attributes, a join selects meaningful cross-relation combinations, and set
operations compare compatible results. Chapter 3 expresses these operations in SQL.


## After-Class Continuation

Recalculate one algebra expression after changing a tuple in the sample data, and explain
which intermediate relation changed. Assignment, rename, and formal equivalence proofs
are optional extensions rather than required Exam 1 derivations.


## Build and Inspect the Chapter Database

The executable cells use SQLite through Python's standard `sqlite3` module. Follow the
cells in order:

1. Open a database connection and enable foreign-key enforcement.
2. Execute the chapter's `CREATE TABLE` statements before inserting rows.
3. Load the synthetic example data.
4. Inspect the resulting tables, columns, primary keys, and foreign keys.
5. Check referential integrity before running the chapter queries.
6. Predict each query result, execute it, and explain any difference.

`DATABASE_NAME` is initially `:memory:`, so closing the notebook removes the database.
Change it to a filename such as `chapter_database.db` when you want SQLite to create a
persistent database in the notebook's working directory. Do not switch to a persistent
file until the in-memory version runs successfully from top to bottom.


In [1]:
import sqlite3

print(f"Python {__import__('sys').version.split()[0]}; SQLite {sqlite3.sqlite_version}")
DATABASE_NAME = ":memory:"  # Change to "chapter_database.db" to keep a database file.
connection = sqlite3.connect(DATABASE_NAME, isolation_level=None)
connection.execute("PRAGMA foreign_keys = ON")


def run_sql_script(connection, script, max_rows=20):
    """Execute a SQLite script and display result-producing statements."""
    buffer = ""
    for raw_line in script.splitlines():
        stripped = raw_line.strip()
        if stripped.startswith(".print"):
            message = stripped[len(".print"):].strip().strip("\"'")
            print(f"\n{message}")
            continue
        buffer += raw_line + "\n"
        if not sqlite3.complete_statement(buffer):
            continue
        statement = buffer.strip()
        buffer = ""
        if not statement:
            continue
        cursor = connection.execute(statement)
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            print(" | ".join(columns))
            for row in rows[:max_rows]:
                print(" | ".join("NULL" if value is None else str(value) for value in row))
            if len(rows) > max_rows:
                print(f"... additional rows omitted after {max_rows}")
    remaining = "\n".join(
        line for line in buffer.splitlines() if not line.strip().startswith("--")
    ).strip()
    if remaining:
        raise ValueError("The embedded SQL ends with an incomplete statement.")


def inspect_database(connection):
    """Display tables, columns, primary keys, foreign keys, and integrity status."""
    tables = [
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name"
        )
    ]
    print("Tables:", ", ".join(tables) if tables else "none")
    for table in tables:
        columns = connection.execute(f'PRAGMA table_info("{table}")').fetchall()
        primary_key = [row[1] for row in sorted(columns, key=lambda row: row[5]) if row[5]]
        print(f"\n{table}")
        print("  columns:", ", ".join(f"{row[1]} {row[2]}" for row in columns))
        print("  primary key:", ", ".join(primary_key) if primary_key else "none")
        for index_row in connection.execute(f'PRAGMA index_list("{table}")').fetchall():
            if index_row[2] and index_row[3] == "u":
                unique_columns = [
                    row[2]
                    for row in connection.execute(
                        f'PRAGMA index_info("{index_row[1]}")'
                    ).fetchall()
                ]
                print("  unique constraint:", ", ".join(unique_columns))
        foreign_keys = connection.execute(f'PRAGMA foreign_key_list("{table}")').fetchall()
        for foreign_key in foreign_keys:
            print(f"  foreign key: {foreign_key[3]} -> {foreign_key[2]}.{foreign_key[4]}")
    violations = connection.execute("PRAGMA foreign_key_check").fetchall()
    print("\nForeign-key check:", "PASS" if not violations else violations)


Python 3.12.13; SQLite 3.53.1


### Course-registration setup


In [2]:
SQL_1 = """PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS enrollment;
DROP TABLE IF EXISTS course;
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS department;

CREATE TABLE department (
    dept_code TEXT PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    building TEXT NOT NULL
);

CREATE TABLE student (
    student_id TEXT PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    student_name TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE course (
    course_id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE enrollment (
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL,
    term TEXT NOT NULL,
    grade TEXT,
    PRIMARY KEY (student_id, course_id, term),
    FOREIGN KEY (student_id) REFERENCES student (student_id),
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);

INSERT INTO department (dept_code, dept_name, building) VALUES
    ('DES', 'Digital Design', 'Hong Hall'),
    ('FIN', 'Finance', 'Cheng Hall'),
    ('IM', 'Information Management', 'Hong Hall');

INSERT INTO student (student_id, email, student_name, dept_code) VALUES
    ('S101', 'an.chen@example.edu', 'An Chen', 'IM'),
    ('S102', 'bea.lin@example.edu', 'Bea Lin', 'FIN'),
    ('S103', 'kai.wu@example.edu', 'Kai Wu', 'IM'),
    ('S104', 'mira.ho@example.edu', 'Mira Ho', 'DES');

INSERT INTO course (course_id, title, dept_code, credits) VALUES
    ('DB201', 'Database Management', 'IM', 3),
    ('FT210', 'Financial Technology', 'FIN', 3),
    ('ML230', 'Machine Learning', 'IM', 3),
    ('WD120', 'Web Design', 'DES', 2);

INSERT INTO enrollment (student_id, course_id, term, grade) VALUES
    ('S101', 'DB201', '115-1', 'A'),
    ('S101', 'FT210', '115-1', 'B+'),
    ('S102', 'FT210', '115-1', 'A-'),
    ('S103', 'DB201', '115-1', 'B'),
    ('S103', 'ML230', '115-1', 'A'),
    ('S104', 'WD120', '115-1', 'A-');
"""


In [3]:
run_sql_script(connection, SQL_1)


### Inspect the Database You Created

Read the output as a schema check: confirm the table names, column types, primary-key order, foreign-key direction, and integrity result.


In [4]:
inspect_database(connection)


Tables: course, department, enrollment, student

course
  columns: course_id TEXT, title TEXT, dept_code TEXT, credits INTEGER
  primary key: course_id
  foreign key: dept_code -> department.dept_code

department
  columns: dept_code TEXT, dept_name TEXT, building TEXT
  primary key: dept_code
  unique constraint: dept_name

enrollment
  columns: student_id TEXT, course_id TEXT, term TEXT, grade TEXT
  primary key: student_id, course_id, term
  foreign key: course_id -> course.course_id
  foreign key: student_id -> student.student_id

student
  columns: student_id TEXT, email TEXT, student_name TEXT, dept_code TEXT
  primary key: student_id
  unique constraint: email
  foreign key: dept_code -> department.dept_code

Foreign-key check: PASS


### Relational-model lab


In [5]:
SQL_2 = """-- Chapter 2 executable examples
-- DBMS used for this file: SQLite
-- Run course_registration_setup.sql first.
-- ORDER BY is used only to make displayed output reproducible. A formal
-- relation has no tuple order.

-- Example 1: display a relation instance.
SELECT student_id, email, student_name, dept_code
FROM student
ORDER BY student_id;

-- Example 2: selection, sigma dept_code = 'IM' (student).
SELECT student_id, email, student_name, dept_code
FROM student
WHERE dept_code = 'IM'
ORDER BY student_id;

-- Example 3: projection, Pi dept_code (student).
-- DISTINCT is required because formal relational-algebra projection removes
-- duplicate tuples, while SQL does not remove them unless requested.
SELECT DISTINCT dept_code
FROM student
ORDER BY dept_code;

-- Example 4: composition, Pi student_name (sigma dept_code = 'IM' (student)).
SELECT student_name
FROM student
WHERE dept_code = 'IM'
ORDER BY student_name;

-- Example 5: Cartesian product of two two-tuple relations.
WITH selected_students AS (
    SELECT student_id
    FROM student
    WHERE student_id IN ('S101', 'S102')
),
selected_courses AS (
    SELECT course_id
    FROM course
    WHERE course_id IN ('DB201', 'FT210')
)
SELECT selected_students.student_id, selected_courses.course_id
FROM selected_students CROSS JOIN selected_courses
ORDER BY selected_students.student_id, selected_courses.course_id;

-- Example 6: theta join between student and enrollment.
SELECT student.student_name, enrollment.course_id, enrollment.term
FROM student
JOIN enrollment ON student.student_id = enrollment.student_id
ORDER BY student.student_id, enrollment.course_id;

-- Example 7a: union of students enrolled in DB201 or FT210.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
UNION
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 7b: intersection of students enrolled in both DB201 and FT210.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
INTERSECT
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 7c: set difference, DB201 students who are not in FT210.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
EXCEPT
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 8: assignment expressed with temporary names in a WITH clause.
WITH db_students AS (
    SELECT student_id FROM enrollment WHERE course_id = 'DB201'
),
fintech_students AS (
    SELECT student_id FROM enrollment WHERE course_id = 'FT210'
)
SELECT student_id FROM db_students
INTERSECT
SELECT student_id FROM fintech_students
ORDER BY student_id;

-- Example 9: rename the student relation twice to compare students in the
-- same department. The ID comparison removes self-pairs and reversed pairs.
SELECT s1.student_name AS student_1, s2.student_name AS student_2, s1.dept_code
FROM student AS s1
CROSS JOIN student AS s2
WHERE s1.dept_code = s2.dept_code
  AND s1.student_id < s2.student_id
ORDER BY s1.student_id, s2.student_id;

-- Example 10a: filter after joining.
SELECT student.student_name, enrollment.course_id
FROM student
JOIN enrollment ON student.student_id = enrollment.student_id
WHERE student.dept_code = 'IM'
ORDER BY student.student_id, enrollment.course_id;

-- Example 10b: filter student first, then join. The result must equal 10a.
WITH im_students AS (
    SELECT student_id, student_name
    FROM student
    WHERE dept_code = 'IM'
)
SELECT im_students.student_name, enrollment.course_id
FROM im_students
JOIN enrollment ON im_students.student_id = enrollment.student_id
ORDER BY im_students.student_id, enrollment.course_id;

-- Student practice: write or predict the result before running each query.
-- P1. Select Finance students and retain all student attributes.
-- P2. Project the distinct building values from department.
-- P3. Return the names of students enrolled in DB201.
-- P4. Find students enrolled in FT210 but not DB201.
-- P5. Explain why removing the join condition in P3 changes the result.
"""


In [6]:
run_sql_script(connection, SQL_2)


student_id | email | student_name | dept_code
S101 | an.chen@example.edu | An Chen | IM
S102 | bea.lin@example.edu | Bea Lin | FIN
S103 | kai.wu@example.edu | Kai Wu | IM
S104 | mira.ho@example.edu | Mira Ho | DES
student_id | email | student_name | dept_code
S101 | an.chen@example.edu | An Chen | IM
S103 | kai.wu@example.edu | Kai Wu | IM
dept_code
DES
FIN
IM
student_name
An Chen
Kai Wu
student_id | course_id
S101 | DB201
S101 | FT210
S102 | DB201
S102 | FT210
student_name | course_id | term
An Chen | DB201 | 115-1
An Chen | FT210 | 115-1
Bea Lin | FT210 | 115-1
Kai Wu | DB201 | 115-1
Kai Wu | ML230 | 115-1
Mira Ho | WD120 | 115-1
student_id
S101
S102
S103
student_id
S101
student_id
S103
student_id
S101
student_1 | student_2 | dept_code
An Chen | Kai Wu | IM
student_name | course_id
An Chen | DB201
An Chen | FT210
Kai Wu | DB201
Kai Wu | ML230
student_name | course_id
An Chen | DB201
An Chen | FT210
Kai Wu | DB201
Kai Wu | ML230


### Reproducibility Check


In [7]:
assert connection.execute("SELECT COUNT(*) FROM department").fetchone()[0] == 3
assert connection.execute("SELECT COUNT(*) FROM student").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM course").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM enrollment").fetchone()[0] == 6
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
print("Notebook checks passed.")


Notebook checks passed.


In [8]:
connection.close()
print("In-memory database closed.")


In-memory database closed.
